# Tesis MIA 303 - Fase 2: Puente de deteccion MIMIC <-> Anexo 2

**Autor:** Carlos Perez Perez - **Curso:** MIA 303 - **Mayo 2026**

---

Este notebook construye el **puente** entre dos mundos:

| | |
|---|---|
| **Notas clinicas de MIMIC-IV** | epicrisis (`discharge`) escritas en **ingles** |
| **Anexo 2 de la Directiva EsSalud** | catalogo de 231 eventos adversos en **espanol**, ya valorados por panel experto |

El objetivo de la Fase 2 es **detectar** eventos adversos en el texto de las notas y **codificarlos**
segun el `id_evento` del Anexo 2, para luego aplicar la Matriz de Priorizacion GEMSES.

Este notebook implementa una primera version por **weak supervision** (reglas de palabras clave),
que servira como base del corpus etiquetado de la Fase 1 del roadmap.

> **Marco etico:** se usa MIMIC-IV-Note v2.2 (DOI 10.13026/1n74-ne17) bajo DUA firmado.
> No se redistribuyen datos ni se intenta re-identificar pacientes.


## 1. Configuracion e importaciones

In [1]:
import duckdb
import pandas as pd
import json, re
from pathlib import Path

print("DuckDB :", duckdb.__version__)
print("Pandas :", pd.__version__)


DuckDB : 1.5.2
Pandas : 2.3.3


In [2]:
# --- Rutas (ajusta solo si tus carpetas son distintas) ---
PATH_MIMIC_NOTE = Path(r"C:\MIMIC\note\note")            # epicrisis MIMIC
PATH_GEMSES_DB  = Path(r"C:\BASE DATOS\db\gemses.db")     # base con el Anexo 2 corregido
PATH_MAPEO      = Path("mapeo_anexo2_ingles.json")       # diccionario ES->EN (en esta carpeta)
PATH_WORK       = Path(r"C:\MIMIC\tesis")                # carpeta de salida

DISCHARGE = PATH_MIMIC_NOTE / "discharge.csv.gz"

for label, p in [("Notas MIMIC", DISCHARGE), ("gemses.db", PATH_GEMSES_DB), ("Mapeo ES->EN", PATH_MAPEO)]:
    estado = "[OK]   " if p.exists() else "[FALTA]"
    print(f"  {estado} {label:15s} {p}")


  [OK]    Notas MIMIC     C:\MIMIC\note\note\discharge.csv.gz
  [OK]    gemses.db       C:\BASE DATOS\db\gemses.db
  [OK]    Mapeo ES->EN    mapeo_anexo2_ingles.json


## 2. Cargar el Anexo 2 (catalogo de eventos adversos)

El Anexo 2 ya esta corregido y cargado en `gemses.db` (231 eventos, 12 naturalezas, codificacion UTF-8).
La columna `impacto` viene **tal cual de la directiva**; `verificacion` marca los 33 eventos cuyo
impacto no cuadra con la formula GEMSES estandar.

In [3]:
con_gem = duckdb.connect(str(PATH_GEMSES_DB), read_only=True)
anexo2 = con_gem.execute("SELECT * FROM anexo2_eventos_adversos").df()
con_gem.close()

print(f"Anexo 2: {len(anexo2)} eventos - {anexo2['naturaleza'].nunique()} naturalezas")
print(f"Eventos marcados REVISAR: {(anexo2['verificacion']=='REVISAR').sum()}")

resumen_nat = (anexo2.groupby('naturaleza')
               .agg(n_eventos=('id_evento','count'),
                    impacto_prom=('impacto','mean'),
                    impacto_max=('impacto','max'))
               .round(2).sort_values('n_eventos', ascending=False))
resumen_nat


Anexo 2: 231 eventos - 12 naturalezas
Eventos marcados REVISAR: 33


,n_eventos,impacto_prom,impacto_max
naturaleza,,,
INFECCIÓN ASOCIADA A LA ATENCIÓN,49,4.62,8.55
PROCEDIMIENTO,34,5.78,8.55
MEDICACIÓN,33,4.67,8.55
HISTORIA CLÍNICA,23,2.08,3.70
GESTIÓN DE LA ORGANIZACIÓN,21,4.05,7.35
SANGRE/HEMODERIVADOS,19,5.69,7.35
CUIDADO DEL PACIENTE,19,2.31,6.75
COMPORTAMIENTO,14,1.27,3.25
DISPOSITIVO MÉDICO / EQUIPO / BIEN,9,2.58,4.50


## 3. El problema del puente: espanol <-> ingles

El Anexo 2 esta en espanol; las notas de MIMIC en ingles. No se puede cruzar directamente por texto.

La solucion de esta fase es un **diccionario de mapeo** (`mapeo_anexo2_ingles.json`): para cada evento
del Anexo 2 razonablemente detectable en una epicrisis, se definen los **patrones en ingles clinico**
que delatan su presencia.

```
  Anexo 2 (ES)                  Patrones (EN)               Nota MIMIC
  -------------                 -------------               ----------
  202 "Caida de paciente"   ->  "fell", "found on floor" -> "...patient fell in
                                                              the bathroom..."
  601 "Infeccion torrente   ->  "bacteremia", "CLABSI"   -> "...developed
       sanguineo"                                            MRSA bacteremia..."
```

Cada patron tiene un nivel de **confianza** (alta / media / baja) segun su riesgo de falso positivo.
Esto es una *labeling function* clasica de **weak supervision**.

## 4. Cargar el diccionario de mapeo

In [4]:
with open(PATH_MAPEO, encoding='utf-8') as f:
    mapeo_raw = json.load(f)

mapeo = pd.DataFrame(mapeo_raw['mapeo'])
mapeo['n_patrones'] = mapeo['patrones_en'].str.len()

print(f"Eventos del Anexo 2 cubiertos por el mapeo inicial: {len(mapeo)} de {len(anexo2)}")
print(f"Total de patrones de deteccion: {mapeo['n_patrones'].sum()}")
print()
print("Cobertura por naturaleza:")
print(mapeo.groupby('naturaleza').agg(eventos_cubiertos=('id_evento','count'),
                                      patrones=('n_patrones','sum')).to_string())
print()
print("Distribucion por nivel de confianza:")
print(mapeo['confianza'].value_counts().to_string())


Eventos del Anexo 2 cubiertos por el mapeo inicial: 51 de 231
Total de patrones de deteccion: 186

Cobertura por naturaleza:
                                  eventos_cubiertos  patrones
naturaleza                                                   
COMPORTAMIENTO                                    2        10
CUIDADO DEL PACIENTE                             10        41
DIAGNÓSTICO                                       2         8
INFECCIÓN ASOCIADA A LA ATENCIÓN                 15        46
MEDICACIÓN                                       11        40
PROCEDIMIENTO                                     9        33
SANGRE/HEMODERIVADOS                              2         8

Distribucion por nivel de confianza:
confianza
media    26
alta     21
baja      4


## 5. Cargar una muestra de notas de MIMIC

Para el desarrollo trabajamos con una muestra reproducible (seed fijo). En produccion se aplicara
al corpus completo de 331,793 epicrisis.

In [5]:
con_mimic = duckdb.connect()

N_MUESTRA = 2000  # subir a 5000 para el corpus de la Fase 1

notas = con_mimic.execute(f'''
    SELECT note_id, subject_id, hadm_id, charttime, text
    FROM read_csv_auto('{DISCHARGE.as_posix()}', compression='gzip')
    USING SAMPLE {N_MUESTRA} ROWS (reservoir, 42)
''').df()
con_mimic.close()

notas['text_lower'] = notas['text'].str.lower()
print(f"Muestra cargada: {len(notas)} epicrisis")
print(f"Longitud media: {notas['text'].str.len().mean():.0f} caracteres")


Muestra cargada: 2000 epicrisis
Longitud media: 10541 caracteres


## 6. Weak supervision: funcion de deteccion

La funcion recorre cada nota y, para cada evento del mapeo, busca sus patrones.
Devuelve una fila por cada (nota, evento) detectado, con el patron que disparo la coincidencia.

Los patrones se compilan con limites de palabra (`\b`) para evitar coincidencias parciales
(ej. que "fall" no dispare dentro de "football").

In [6]:
# Pre-compilar los patrones en regex
patrones_compilados = []
for _, fila in mapeo.iterrows():
    for patron in fila['patrones_en']:
        rx = re.compile(r'\b' + re.escape(patron.lower()) + r'\b')
        patrones_compilados.append((fila['id_evento'], fila['evento'],
                                    fila['naturaleza'], fila['confianza'], patron, rx))

print(f"{len(patrones_compilados)} patrones compilados.")

def detectar_eventos(df_notas):
    '''Aplica weak supervision: una fila por cada (nota, evento) detectado.'''
    hits = []
    for _, nota in df_notas.iterrows():
        txt = nota['text_lower']
        vistos = set()
        for id_ev, evento, nat, conf, patron, rx in patrones_compilados:
            if id_ev in vistos:
                continue  # un evento se cuenta una sola vez por nota
            if rx.search(txt):
                hits.append({
                    'note_id': nota['note_id'], 'subject_id': nota['subject_id'],
                    'hadm_id': nota['hadm_id'], 'id_evento': id_ev, 'evento': evento,
                    'naturaleza': nat, 'confianza': conf, 'patron_disparado': patron,
                })
                vistos.add(id_ev)
    return pd.DataFrame(hits)


186 patrones compilados.


In [7]:
detecciones = detectar_eventos(notas)

n_notas_con_evento = detecciones['note_id'].nunique()
print(f"Notas analizadas        : {len(notas)}")
print(f"Notas con >=1 evento    : {n_notas_con_evento} ({100*n_notas_con_evento/len(notas):.1f}%)")
print(f"Detecciones totales     : {len(detecciones)}")
print(f"Eventos distintos       : {detecciones['id_evento'].nunique()}")


Notas analizadas        : 2000
Notas con >=1 evento    : 1358 (67.9%)
Detecciones totales     : 2500
Eventos distintos       : 35


## 7. Que eventos se detectaron y con que frecuencia

In [8]:
frec = (detecciones.groupby(['id_evento','evento','naturaleza','confianza'])
        .size().reset_index(name='frecuencia')
        .sort_values('frecuencia', ascending=False))
print("Top 20 eventos detectados en la muestra:")
frec.head(20)


Top 20 eventos detectados en la muestra:


,id_evento,evento,naturaleza,confianza,frecuencia
2,202,Caída de paciente,CUIDADO DEL PACIENTE,media,427
33,8032,Trombosis venosa,PROCEDIMIENTO,media,425
29,8016,Neumotórax,PROCEDIMIENTO,media,305
5,602,Infección del tracto urinario,INFECCIÓN ASOCIADA A LA ATENCIÓN,media,282
18,6022,Infección de la piel,INFECCIÓN ASOCIADA A LA ATENCIÓN,media,160
27,7031,Diarrea asociada a antimicrobianos,MEDICACIÓN,alta,120
4,601,Infección del torrente sanguíneo (ITS),INFECCIÓN ASOCIADA A LA ATENCIÓN,alta,116
19,6030,Osteomielitis,INFECCIÓN ASOCIADA A LA ATENCIÓN,alta,84
7,705,Dosis o frecuencia errónea,MEDICACIÓN,media,81
11,2011,Lesiones de piel,CUIDADO DEL PACIENTE,media,73


## 8. Conectar con la Matriz GEMSES: Impacto de lo detectado

Cruzamos los eventos detectados con el Anexo 2 para traer su **Impacto** (G) - la valoracion del
panel experto - y calcular el **Indice de Frecuencia** (B = A / total) sobre la muestra.

Primer eslabon de la Matriz de Priorizacion:

```
   B (Indice Frecuencia)  x  G (Impacto)  =  H (Riesgo)
   I = H / suma(H)        J = I x 100     -> Verde / Amarillo / Rojo por percentiles
```

In [9]:
total_detecciones = len(detecciones)
matriz = frec.merge(anexo2[['id_evento','impacto','impacto_formula','verificacion']],
                    on='id_evento', how='left')
matriz = matriz.rename(columns={'frecuencia':'A_frecuencia', 'impacto':'G_impacto'})
matriz['B_indice_frecuencia'] = (matriz['A_frecuencia'] / total_detecciones).round(4)
matriz['H_riesgo'] = (matriz['B_indice_frecuencia'] * matriz['G_impacto']).round(4)

suma_h = matriz['H_riesgo'].sum()
matriz['I_indice_impacto'] = (matriz['H_riesgo'] / suma_h).round(4)
matriz['J_nivel_gestion'] = (matriz['I_indice_impacto'] * 100).round(2)

q2, q3 = matriz['J_nivel_gestion'].quantile([0.50, 0.75])
def clasificar(j):
    if j >= q3:  return 'ROJO (Director)'
    if j >= q2:  return 'AMARILLO (Departamento)'
    return 'VERDE (Servicio)'
matriz['nivel'] = matriz['J_nivel_gestion'].apply(clasificar)

cols_show = ['id_evento','evento','naturaleza','A_frecuencia','B_indice_frecuencia',
             'G_impacto','H_riesgo','I_indice_impacto','J_nivel_gestion','nivel','verificacion']
print("Matriz de Priorizacion GEMSES sobre la muestra (top 15 por Riesgo):")
matriz.sort_values('H_riesgo', ascending=False)[cols_show].head(15)


Matriz de Priorizacion GEMSES sobre la muestra (top 15 por Riesgo):


,id_evento,evento,naturaleza,A_frecuencia,B_indice_frecuencia,G_impacto,H_riesgo,I_indice_impacto,J_nivel_gestion,nivel,verificacion
1,8032,Trombosis venosa,PROCEDIMIENTO,425,0.1700,3.30,0.5610,0.1468,14.68,ROJO (Director),OK
2,8016,Neumotórax,PROCEDIMIENTO,305,0.1220,4.30,0.5246,0.1373,13.73,ROJO (Director),OK
3,602,Infección del tracto urinario,INFECCIÓN ASOCIADA A LA ATENCIÓN,282,0.1128,4.50,0.5076,0.1329,13.29,ROJO (Director),OK
7,6030,Osteomielitis,INFECCIÓN ASOCIADA A LA ATENCIÓN,84,0.0336,8.55,0.2873,0.0752,7.52,ROJO (Director),OK
5,7031,Diarrea asociada a antimicrobianos,MEDICACIÓN,120,0.0480,5.70,0.2736,0.0716,7.16,ROJO (Director),OK
0,202,Caída de paciente,CUIDADO DEL PACIENTE,427,0.1708,1.35,0.2306,0.0604,6.04,ROJO (Director),OK
6,601,Infección del torrente sanguíneo (ITS),INFECCIÓN ASOCIADA A LA ATENCIÓN,116,0.0464,4.50,0.2088,0.0547,5.47,ROJO (Director),OK
10,6037,Endocarditis,INFECCIÓN ASOCIADA A LA ATENCIÓN,69,0.0276,7.35,0.2029,0.0531,5.31,ROJO (Director),OK
12,6035,Meningitis o ventriculitis,INFECCIÓN ASOCIADA A LA ATENCIÓN,43,0.0172,8.55,0.1471,0.0385,3.85,ROJO (Director),OK
4,6022,Infección de la piel,INFECCIÓN ASOCIADA A LA ATENCIÓN,160,0.0640,2.05,0.1312,0.0343,3.43,AMARILLO (Departamento),OK


In [10]:
print("Distribucion por Nivel de Gestion:")
print(matriz['nivel'].value_counts().to_string())
print()
print("Eventos detectados que estan en la lista REVISAR (impacto a confirmar con la directiva):")
rev = matriz[matriz['verificacion']=='REVISAR'][['id_evento','evento','G_impacto','impacto_formula']]
print(rev.to_string(index=False) if len(rev) else "  (ninguno en esta muestra)")


Distribucion por Nivel de Gestion:
nivel
VERDE (Servicio)           17
ROJO (Director)             9
AMARILLO (Departamento)     9

Eventos detectados que estan en la lista REVISAR (impacto a confirmar con la directiva):
 id_evento                evento  G_impacto  impacto_formula
      6049 Sospecha de Infección       5.10             4.65
      9017      Reacción adversa       7.35             6.75


## 9. Guardar el corpus pre-etiquetado

In [11]:
out_detecciones = PATH_WORK / "fase2_detecciones_weak_supervision.csv"
detecciones.to_csv(out_detecciones, index=False, encoding='utf-8-sig')

out_matriz = PATH_WORK / "fase2_matriz_gemses_muestra.csv"
matriz.to_csv(out_matriz, index=False, encoding='utf-8-sig')

print(f"Guardado: {out_detecciones.name}  ({len(detecciones)} filas)")
print(f"Guardado: {out_matriz.name}  ({len(matriz)} eventos)")
print()
print("RECORDATORIO: estos archivos contienen datos derivados de MIMIC bajo DUA.")
print("NO subir a GitHub (ya cubiertos por .gitignore).")


Guardado: fase2_detecciones_weak_supervision.csv  (2500 filas)
Guardado: fase2_matriz_gemses_muestra.csv  (35 eventos)

RECORDATORIO: estos archivos contienen datos derivados de MIMIC bajo DUA.
NO subir a GitHub (ya cubiertos por .gitignore).


## 10. Proximos pasos

Quedo construido el **puente MIMIC <-> Anexo 2** y un primer corpus pre-etiquetado por weak supervision.

**Limitaciones de esta version (a mejorar):**

1. **Cobertura parcial** - el mapeo cubre ~49 de 231 eventos. Faltan los eventos administrativos
   (Gestion, Historia Clinica) que casi no aparecen en epicrisis clinicas.
2. **Sin negacion ni contexto** - un patron como "no evidence of pneumonia" se cuenta como positivo.
   La Fase 2 NLP con BioBERT/ClinicalBERT debe resolver esto.
3. **Patrones de confianza baja/media** - necesitan validacion manual.

**Siguiente notebook - `fase2_nlp_bioBERT.ipynb`:**

1. Usar `fase2_detecciones_weak_supervision.csv` como etiquetas debiles iniciales.
2. Fine-tunear BioBERT/ClinicalBERT para NER de eventos adversos (con manejo de negacion).
3. Comparar la deteccion por reglas vs. la del modelo.
4. Ampliar el mapeo `mapeo_anexo2_ingles.json` con los hallazgos.

> Antes de avanzar: hacer un **spot check manual** de 20 notas detectadas para medir falsos positivos.
> Esa es la metrica de precision inicial que vas a citar en el Capitulo III.
